# ParliamentLens — Data Audit Notebook

This notebook is the **first validation step** before feature engineering or dashboard building.

The goal is to understand the raw datasets and verify that our loading and cleaning code behaves as expected. We are **not** building metrics here yet. We are checking:
- file loading
- dataframe shapes and columns
- missing values
- duplicates
- repeated text patterns
- Unix timestamp parsing into datetime fields

This notebook is intentionally exploratory, but it follows the same project modules we created in `src/data/` so the logic stays reusable.

## 1. Imports and notebook setup

We import the project modules we already wrote:
- `loader.py` to load the raw files
- `cleaning.py` to apply safe technical cleaning
- `validation.py` to inspect the data

The small `sys.path` step allows the notebook to import from the project root even when it is opened from inside the `notebooks/` folder.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.loader import (
    load_bills,
    load_transcripts_topic,
    load_transcripts_text,
    load_voting_sessions,
    load_legislators,
    load_topics,
    print_dataframe_summary,
)
from src.data.cleaning import (
    clean_dataframe_basic, 
    filter_transcript_parties, 
    add_placeholder_flag,
)
from src.data.validation import (
    summarize_dataframe,
    missing_values_summary,
    duplicate_summary,
    value_counts_summary,
    text_value_profile,
    datetime_parse_summary,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

## 2. Load the core datasets

We start by loading the main datasets with `verbose=True` so we can see their shape and approximate memory usage.

For the first audit, it is okay to load them one by one. If the full text transcript file feels too heavy later, we can keep it separate from the rest of the dashboard workflow.

In [2]:
bills_df = load_bills(verbose=True)
transcripts_topic_df = load_transcripts_topic(verbose=True)
transcripts_text_df = load_transcripts_text(verbose=True)
voting_df = load_voting_sessions(verbose=True)

legislators_df = load_legislators(verbose=True)
topics_df = load_topics(verbose=True)

ca_fed_bills_42-43.csv: shape=(728, 15), columns=15, memory=1.27 MB
ca_fed_bills_44.csv: shape=(412, 15), columns=15, memory=620.84 KB
bills (combined): shape=(1140, 15), columns=15, memory=1.88 MB
transcripts_42_parliament.csv: shape=(121100, 9), columns=9, memory=86.28 MB
transcripts_43_parliament.csv: shape=(52802, 9), columns=9, memory=36.07 MB
transcripts_44_parliament.csv: shape=(130898, 9), columns=9, memory=89.22 MB
transcripts_topic (combined): shape=(304800, 9), columns=9, memory=211.57 MB
transcript_43_with_text.csv: shape=(56424, 10), columns=10, memory=128.30 MB
transcript_44_with_text.csv: shape=(130898, 10), columns=10, memory=293.48 MB
transcripts_text (combined): shape=(187322, 10), columns=10, memory=421.78 MB
ca_fed_house_vote_session_42-43.csv: shape=(319, 11), columns=11, memory=15.27 MB
ca_fed_house_vote_session_44.csv: shape=(481, 11), columns=11, memory=24.58 MB
voting_sessions (combined): shape=(800, 11), columns=11, memory=39.86 MB
legislators: shape=(250, 12)

## 3. Apply safe cleaning

We only apply the conservative cleaning rules we agreed on:
- trim string whitespace
- convert truly empty strings to `pd.NA`
- parse Unix timestamp columns like `time` into new datetime columns such as `time_dt`

We keep the original raw columns intact.

In [3]:
voting_clean = clean_dataframe_basic(voting_df, unix_time_columns=["time"])

legislators_clean = clean_dataframe_basic(legislators_df)
topics_clean = clean_dataframe_basic(topics_df)

In [4]:
bills_clean = clean_dataframe_basic(bills_df, unix_time_columns=["time"], list_like_columns=["level_2_topics"])
bills_clean = add_placeholder_flag(
    bills_clean,
    source_column="summary",
    placeholder_values=["Bill summary not available"],
)

In [5]:
transcripts_topic_clean = clean_dataframe_basic(
    transcripts_topic_df,
    unix_time_columns=["time"],
    list_like_columns=["level_3_topics", "level_2_topics"]
)
transcripts_topic_clean = filter_transcript_parties(transcripts_topic_clean)

transcripts_text_clean = clean_dataframe_basic(
    transcripts_text_df,
    unix_time_columns=["time"],
    list_like_columns=["level_3_topics", "level_2_topics"]
)
transcripts_text_clean = filter_transcript_parties(transcripts_text_clean)

## 4. Dataset-level summaries

This gives a compact overview of each dataset:
- number of rows
- number of columns
- duplicate row count

It is a good first sanity check before looking at individual fields.

In [6]:
dataset_summaries = pd.concat(
    [
        summarize_dataframe(bills_clean, name="Bills"),
        summarize_dataframe(transcripts_topic_clean, name="Transcripts (topic only)"),
        summarize_dataframe(transcripts_text_clean, name="Transcripts (with text)"),
        summarize_dataframe(voting_clean, name="Voting sessions"),
        summarize_dataframe(legislators_clean, name="Legislators"),
        summarize_dataframe(topics_clean, name="Topics"),
    ],
    ignore_index=True,
)

dataset_summaries

,dataset,rows,columns,duplicate_rows
0,Bills,1140,17,0
1,Transcripts (topic only),304800,12,4092
2,Transcripts (with text),183716,13,138
3,Voting sessions,800,11,0
4,Legislators,250,12,0
5,Topics,44653,5,0


## 5. Column inspection

Before we compute any features, we should confirm the actual schema of each loaded dataset.

This is especially useful because some files may come from slightly different export pipelines across parliaments.

In [7]:
for name, df in {
    "Bills": bills_clean,
    "Transcripts (topic only)": transcripts_topic_clean,
    "Transcripts (with text)": transcripts_text_clean,
    "Voting sessions": voting_clean,
    "Legislators": legislators_clean,
    "Topics": topics_clean,
}.items():
    print(f"\n{name}")
    print("-" * len(name))
    print(df.columns.tolist())


Bills
-----
['source_id', 'name', 'number', 'parliament', 'session', 'status', 'stage', 'url', 'sponsor_name', 'sponsor_id', 'topics', 'summary', 'level_2_topics', 'source_file', 'dataset_name', 'level_2_topics_list', 'summary_is_placeholder']

Transcripts (topic only)
------------------------
['speaker', 'constituency', 'party', 'level_3_topics', 'level_2_topics', 'parliament', 'time', 'source_file', 'dataset_name', 'time_dt', 'level_3_topics_list', 'level_2_topics_list']

Transcripts (with text)
-----------------------
['name', 'constituency', 'party', 'level_3_topics', 'level_2_topics', 'parliament', 'time', 'text', 'source_file', 'dataset_name', 'time_dt', 'level_3_topics_list', 'level_2_topics_list']

Voting sessions
---------------
['date', 'result', 'title', 'bill_number', 'bill_title', 'parliament', 'session', 'url', 'vote_data', 'source_file', 'dataset_name']

Legislators
-----------
['area', 'division', 'email', 'id', 'image_url', 'legislature', 'name', 'party', 'role', 'sou

## 6. Missing value summaries

We inspect missingness by dataset to understand where the main data gaps are.

This is important for:
- deciding which fields are reliable enough for KPI use
- checking whether text fields like `summary` or `text` are incomplete
- understanding how usable the API reference files are

In [8]:
missing_values_summary(bills_clean).head(20)

,column,missing_count,missing_pct
0,dataset_name,0,0.0
1,level_2_topics,0,0.0
2,level_2_topics_list,0,0.0
3,name,0,0.0
4,number,0,0.0
5,parliament,0,0.0
6,session,0,0.0
7,source_file,0,0.0
8,source_id,0,0.0
9,sponsor_id,0,0.0


In [9]:
missing_values_summary(transcripts_topic_clean).head(20)

,column,missing_count,missing_pct
0,level_3_topics_list,20271,6.65
1,constituency,0,0.00
2,dataset_name,0,0.00
3,level_2_topics,0,0.00
4,level_2_topics_list,0,0.00
5,level_3_topics,0,0.00
6,parliament,0,0.00
7,party,0,0.00
8,source_file,0,0.00
9,speaker,0,0.00


In [10]:
missing_values_summary(transcripts_text_clean).head(20)

,column,missing_count,missing_pct
0,level_3_topics_list,12742,6.94
1,text,62,0.03
2,constituency,0,0.00
3,dataset_name,0,0.00
4,level_2_topics,0,0.00
5,level_2_topics_list,0,0.00
6,level_3_topics,0,0.00
7,name,0,0.00
8,parliament,0,0.00
9,party,0,0.00


In [11]:
missing_values_summary(voting_clean).head(20)

,column,missing_count,missing_pct
0,bill_number,0,0.0
1,bill_title,0,0.0
2,dataset_name,0,0.0
3,date,0,0.0
4,parliament,0,0.0
5,result,0,0.0
6,session,0,0.0
7,source_file,0,0.0
8,title,0,0.0
9,url,0,0.0


## 7. Duplicate checks

We check duplicate rows both broadly and, where useful later, we can also test business-key duplicates using selected columns.

For now, this gives a quick view of whether any dataset looks unusually repetitive.

In [12]:
duplicate_results = {
    "Bills": duplicate_summary(bills_clean),
    "Transcripts (topic only)": duplicate_summary(transcripts_topic_clean),
    "Transcripts (with text)": duplicate_summary(transcripts_text_clean),
    "Voting sessions": duplicate_summary(voting_clean),
    "Legislators": duplicate_summary(legislators_clean),
    "Topics": duplicate_summary(topics_clean),
}

duplicate_results

{'Bills': {'total_rows': 1140, 'duplicate_rows': 0, 'duplicate_pct': 0.0},
 'Transcripts (topic only)': {'total_rows': 304800,
  'duplicate_rows': 4092,
  'duplicate_pct': 1.34},
 'Transcripts (with text)': {'total_rows': 183716,
  'duplicate_rows': 138,
  'duplicate_pct': 0.08},
 'Voting sessions': {'total_rows': 800,
  'duplicate_rows': 0,
  'duplicate_pct': 0.0},
 'Legislators': {'total_rows': 250, 'duplicate_rows': 0, 'duplicate_pct': 0.0},
 'Topics': {'total_rows': 44653, 'duplicate_rows': 0, 'duplicate_pct': 0.0}}

## 8. Time parsing audit

The raw `time` field appears to be stored as Unix seconds such as `1622749500.0`.

We converted it into a new datetime column `time_dt`. This section checks whether the parsing worked and lets us inspect a few sample rows.

In [13]:
datetime_parse_summary(transcripts_topic_clean, source_column="time", parsed_column="time_dt")

{'source_column_exists': True,
 'parsed_column_exists': True,
 'non_null_source': 304800,
 'non_null_parsed': 304800,
 'parse_success_pct': 100.0}

In [14]:
datetime_parse_summary(transcripts_text_clean, source_column="time", parsed_column="time_dt")

{'source_column_exists': True,
 'parsed_column_exists': True,
 'non_null_source': 183716,
 'non_null_parsed': 183716,
 'parse_success_pct': 100.0}

In [15]:
datetime_parse_summary(voting_clean, source_column="time", parsed_column="time_dt")

{'source_column_exists': False,
 'parsed_column_exists': False,
 'non_null_source': None,
 'non_null_parsed': None,
 'parse_success_pct': None}

In [16]:
transcripts_text_clean[["time", "time_dt"]].head(10)

,time,time_dt
0,1.622750e+09,2021-06-03 19:45:00+00:00
1,1.620678e+09,2021-05-10 20:14:00+00:00
2,1.619467e+09,2021-04-26 19:51:00+00:00
3,1.619467e+09,2021-04-26 19:52:00+00:00
4,1.614266e+09,2021-02-25 15:19:00+00:00
5,1.612544e+09,2021-02-05 17:00:00+00:00
6,1.612468e+09,2021-02-04 19:45:00+00:00
7,1.607631e+09,2020-12-10 20:06:00+00:00
8,1.607545e+09,2020-12-09 20:17:00+00:00
9,1.606248e+09,2020-11-24 20:06:00+00:00


## 9. Categorical value checks

For fields that are central to the dashboard, we should inspect the most common values:
- `status`
- `stage`
- `party`
- `level_2_topics`
- maybe `parliament` and `session`

These checks help us understand the data vocabulary before building filters and charts in Streamlit.

In [17]:
value_counts_summary(bills_clean, "status", top_n=20)

,status,count
0,Outside the Order of Precedence,409
1,Royal assent received,244
2,At second reading in the House of Commons,119
3,At second reading in the Senate,101
4,Bill defeated,100
5,At consideration in committee in the Senate,45
6,Bill not proceeded with,40
7,At report stage in the House of Commons,23
8,Senate bill awaiting first reading in the House of Commons,17
9,At third reading in the Senate,12


In [18]:
value_counts_summary(bills_clean, "stage", top_n=20)

,stage,count
0,First reading,762
1,Royal assent,242
2,Second reading,113
3,Third reading,23


In [19]:
value_counts_summary(transcripts_topic_clean, "party", top_n=20)

,party,count
0,Liberal,133490
1,Conservative,95368
2,NDP,46942
3,Bloc Québécois,21043
4,Green Party,4823
5,Independent,2594
6,Co-operative Commonwealth Federation,391
7,People's Party,149


In [20]:
value_counts_summary(transcripts_topic_clean, "level_2_topics", top_n=20)

,level_2_topics,count
0,[],20309
1,['Other'],4086
2,"['Government Operations', 'Macroeconomics', 'Government Operations', 'Government Operations', 'Government Operations', 'Civil Rights', 'Civil Rights', 'Government Operations', 'Technology']",2387
3,['International Affairs'],2024
4,['Government Operations'],1085
5,"['Macroeconomics', 'Macroeconomics', 'Other']",990
6,"['International Affairs', 'Health', 'Other', 'Health']",818
7,"['Other', 'International Affairs', 'Macroeconomics']",756
8,"['International Affairs', 'Law and Crime', 'Law and Crime', 'Health', 'Civil Rights', 'Law and Crime']",690
9,['Domestic Commerce'],640


In [21]:
value_counts_summary(bills_clean, "level_2_topics", top_n=20)

,level_2_topics,count
0,"['Government Operations', 'Law and Crime', 'Law and Crime']",61
1,"['Government Operations', 'Law and Crime', 'Government Operations']",40
2,"['Government Operations', 'Law and Crime']",36
3,"['Government Operations', 'Macroeconomics', 'Law and Crime']",32
4,"['Government Operations', 'Labor', 'Law and Crime']",28
5,"['Government Operations', 'Law and Crime', 'Culture']",28
6,"['Government Operations', 'Law and Crime', 'Civil Rights']",28
7,"['Government Operations', 'Law and Crime', 'Macroeconomics']",28
8,"['Government Operations', 'Law and Crime', 'Social Welfare']",27
9,"['Government Operations', 'Health', 'Law and Crime']",25


## 10. Text profiling before placeholder rules

This section looks at repeated short values in `summary` and `text`. Short, highly repeated strings are good candidates for placeholder or boilerplate patterns.

In [22]:
text_value_profile(bills_clean, column="summary", top_n=30, max_length=100)

,summary,count,text_length
0,Bill summary not available,911,26


In [29]:
text_value_profile(transcripts_text_clean, column="text", top_n=30, max_length=100)

,text,count,text_length
0,I declare the motion carried.,481,29
1,Is that agreed?Some hon. members: Agreed.,480,41
2,The hon. member.,264,16
3,Call in the members.,167,20
4,Is it agreed?Some hon. members: Agreed.,156,39
5,The hon. parliamentary secretary.,149,33
6,"Madam Speaker, I request a recorded division.",122,45
7,"Mr. Speaker, I request a recorded division.",103,43
8,The hon. Leader of the Opposition.,91,34
9,I declare the motion defeated.,68,30


## 11. Optional spot checks

It is often helpful to look at a few rows directly after the summary tables. This helps connect the profiling results back to the original records.

In [24]:
bills_clean.head(5)

,source_id,name,number,parliament,session,status,stage,url,sponsor_name,sponsor_id,topics,summary,level_2_topics,source_file,dataset_name,level_2_topics_list,summary_is_placeholder
0,8061133,An Act relating to railways,S-1,42,1,Introduced as pro forma bill,First reading,https://www.parl.ca/legisinfo/en/bill/42-1/S-1,Yonah Martin,2813,"['Transportation', 'Legislation']",Bill summary not available,"['Transportation', 'Law and Crime']",ca_fed_bills_42-43.csv,bills,"[Transportation, Law and Crime]",True
1,8260714,An Act to amend the Motor Vehicle Safety Act and to make a consequential amendment to another Act,S-2,42,1,Royal assent received,Royal assent,https://www.parl.ca/legisinfo/en/bill/42-1/S-2,Peter Harder,206019,"['Transportation', 'Government Operations', 'Policy']",The pre-release version of this Legislative Summary is now available. Parliamentarians and their staff can obtain a copy by submitting a request or contacting the Library of Parliament. Members of...,"['Transportation', 'Government Operations', 'Government Operations']",ca_fed_bills_42-43.csv,bills,"[Transportation, Government Operations, Government Operations]",False
2,8532485,An Act to amend the Indian Act in response to the Superior Court of Quebec decision in Descheneaux c. Canada (Procureur général),S-3,42,1,Royal assent received,Royal assent,https://www.parl.ca/legisinfo/en/bill/42-1/S-3,Peter Harder,206019,"['Government', 'Legislation', 'Indigenous Affairs']",The pre-release version of this Legislative Summary is now available. Parliamentarians and their staff can obtain a copy by submitting a request or contacting the Library of Parliament. Members of...,"['Government Operations', 'Law and Crime', 'Public Lands']",ca_fed_bills_42-43.csv,bills,"[Government Operations, Law and Crime, Public Lands]",False
3,8560960,An Act to implement a Convention and an Arrangement for the avoidance of double taxation and the prevention of fiscal evasion with respect to taxes on income and to amend an Act in respect of a si...,S-4,42,1,Royal assent received,Royal assent,https://www.parl.ca/legisinfo/en/bill/42-1/S-4,Peter Harder,206019,"['Government', 'Taxation', 'International Agreements']","The Library of Parliament does not prepare Legislative Summaries for bills that implement treaties, conventions, agreements or administrative arrangements bills. The following is a short summary:\...","['Government Operations', 'Social Welfare', 'International Affairs']",ca_fed_bills_42-43.csv,bills,"[Government Operations, Social Welfare, International Affairs]",False
4,8616151,An Act to amend the Tobacco Act and the Non-smokers’ Health Act and to make consequential amendments to other Acts,S-5,42,1,Royal assent received,Royal assent,https://www.parl.ca/legisinfo/en/bill/42-1/S-5,Peter Harder,206019,"['Government', 'Health', 'Legislation']",The pre-release version of this Legislative Summary is now available. Parliamentarians and their staff can obtain a copy by submitting a request or contacting the Library of Parliament. Members of...,"['Government Operations', 'Health', 'Law and Crime']",ca_fed_bills_42-43.csv,bills,"[Government Operations, Health, Law and Crime]",False


In [25]:
transcripts_text_clean.head(5)

,name,constituency,party,level_3_topics,level_2_topics,parliament,time,text,source_file,dataset_name,time_dt,level_3_topics_list,level_2_topics_list
0,Lawrence MacAulay,Cardigan,Liberal,"['Oral questions', 'Peer support', 'Sexual assault', 'Sexual harassment', 'Veterans']","['Other', 'Social Welfare', 'Law and Crime', 'Law and Crime', 'Civil Rights']",43,1.622750e+09,"Mr. Speaker, we thank the Veterans Ombud for her report and agree with her recommendations. We know how important peer support can be for survivors and in budget 2021, we committed to implementing...",transcript_43_with_text.csv,transcripts_text,2021-06-03 19:45:00+00:00,"[Oral questions, Peer support, Sexual assault, Sexual harassment, Veterans]","[Other, Social Welfare, Law and Crime, Law and Crime, Civil Rights]"
1,Lawrence MacAulay,Cardigan,Liberal,"['Books of Remembrance', 'Canadian Forces', 'Statements by Ministers', 'War casualties']","['Other', 'Defense', 'Government Operations', 'Defense']",43,1.620678e+09,"Mr. Speaker, more than 118,000 Canadians and Newfoundlanders have given their lives in service to Canada. We have lost them in the muddy trenches of Flanders, on the shores of the Normandy coast, ...",transcript_43_with_text.csv,transcripts_text,2021-05-10 20:14:00+00:00,"[Books of Remembrance, Canadian Forces, Statements by Ministers, War casualties]","[Other, Defense, Government Operations, Defense]"
2,Lawrence MacAulay,Cardigan,Liberal,"['Communication control', 'Department of Veterans Affairs', 'Desmond, Lionel', 'Homicide', 'Inquiries and public inquiries', 'Mental health', 'Oral questions', 'Suicides', 'Veterans']","['Technology', 'International Affairs', 'Other', 'Law and Crime', 'International Affairs', 'Health', 'Other', 'Health', 'Civil Rights']",43,1.619467e+09,"Mr. Speaker, our hearts go out to the families involved in this tragedy. We have always committed to co-operating fully with the inquiry launched by Nova Scotia. I would like to clarify that the r...",transcript_43_with_text.csv,transcripts_text,2021-04-26 19:51:00+00:00,"[Communication control, Department of Veterans Affairs, Desmond, Lionel, Homicide, Inquiries and public inquiries, Mental health, Oral questions, Suicides, Veterans]","[Technology, International Affairs, Other, Law and Crime, International Affairs, Health, Other, Health, Civil Rights]"
3,Lawrence MacAulay,Cardigan,Liberal,"['Communication control', 'Department of Veterans Affairs', 'Desmond, Lionel', 'Homicide', 'Inquiries and public inquiries', 'Mental health', 'Oral questions', 'Suicides', 'Veterans']","['Technology', 'International Affairs', 'Other', 'Law and Crime', 'International Affairs', 'Health', 'Other', 'Health', 'Civil Rights']",43,1.619467e+09,"Mr. Speaker, I can assure my hon. colleague we are always committed to fully co-operating with the inquiry launched by Nova Scotia. Again, I said we have provided this information to the inquiry f...",transcript_43_with_text.csv,transcripts_text,2021-04-26 19:52:00+00:00,"[Communication control, Department of Veterans Affairs, Desmond, Lionel, Homicide, Inquiries and public inquiries, Mental health, Oral questions, Suicides, Veterans]","[Technology, International Affairs, Other, Law and Crime, International Affairs, Health, Other, Health, Civil Rights]"
4,Lawrence MacAulay,Cardigan,Liberal,"['Atlantic Canada', 'Government bills', 'Introduction and First reading', 'Minister of Human Resources and Skills Development', 'Oil and gas', \""O'Regan, Seamus\"", 'S-3, An Act to amend the Offsho...","['Public Lands', 'Government Operations', 'Education', 'Education', 'Energy', 'Social Welfare', 'International Affairs', 'Law and Crime', 'Health']",43,1.614266e+09,"moved that Bill S-3, An Act to amend the Offshore Health and Safety Act, be read the first time.",transcript_43_with_text.csv,transcripts_text,2021-02-25 15:19:00+00:00,<NA>,"[Public Lands, Government Operations, Education, Education, Energy, Social Welfare, International Affairs, Law and Crime, Health]"


In [26]:
voting_clean.head(5)

,date,result,title,bill_number,bill_title,parliament,session,url,vote_data,source_file,dataset_name
0,2018-6-11,Defeated,"Bill C-69, An Act to enact the Impact Assessment Act and the Canadian Energy Regulator Act, to amend the Navigation Protection Act and to make consequential amendments to other Acts (report stage ...",C-69,"An Act to enact the Impact Assessment Act and the Canadian Energy Regulator Act, to amend the Navigation Protection Act and to make consequential amendments to other Acts",42,1,https://www.ourcommons.ca/members/en/votes/42/1/743,"[{'name': 'Ziad Aboultaif', 'source_id': '89156', 'vote': 'Nay', 'vote_session_url': 'https://www.ourcommons.ca/members/en/votes/42/1/743', 'bill_number': 'C-69'}, {'name': 'Dan Albas', 'source_id...",ca_fed_house_vote_session_42-43.csv,voting_sessions
1,2018-6-11,Defeated,"Bill C-69, An Act to enact the Impact Assessment Act and the Canadian Energy Regulator Act, to amend the Navigation Protection Act and to make consequential amendments to other Acts (report stage ...",C-69,"An Act to enact the Impact Assessment Act and the Canadian Energy Regulator Act, to amend the Navigation Protection Act and to make consequential amendments to other Acts",42,1,https://www.ourcommons.ca/members/en/votes/42/1/744,"[{'name': 'Ziad Aboultaif', 'source_id': '89156', 'vote': 'Yea', 'vote_session_url': 'https://www.ourcommons.ca/members/en/votes/42/1/744', 'bill_number': 'C-69'}, {'name': 'Dan Albas', 'source_id...",ca_fed_house_vote_session_42-43.csv,voting_sessions
2,2018-6-11,Adopted,"Concurrence at report stage of Bill C-69, An Act to enact the Impact Assessment Act and the Canadian Energy Regulator Act, to amend the Navigation Protection Act and to make consequential amendmen...",C-69,"An Act to enact the Impact Assessment Act and the Canadian Energy Regulator Act, to amend the Navigation Protection Act and to make consequential amendments to other Acts",42,1,https://www.ourcommons.ca/members/en/votes/42/1/745,"[{'name': 'Ziad Aboultaif', 'source_id': '89156', 'vote': 'Nay', 'vote_session_url': 'https://www.ourcommons.ca/members/en/votes/42/1/745', 'bill_number': 'C-69'}, {'name': 'Dan Albas', 'source_id...",ca_fed_house_vote_session_42-43.csv,voting_sessions
3,2018-6-11,Defeated,"Bill C-59, An Act respecting national security matters (report stage amendment)",C-59,"National Security Act, 2017",42,1,https://www.ourcommons.ca/members/en/votes/42/1/746,"[{'name': 'Ziad Aboultaif', 'source_id': '89156', 'vote': 'Nay', 'vote_session_url': 'https://www.ourcommons.ca/members/en/votes/42/1/746', 'bill_number': 'C-59'}, {'name': 'Dan Albas', 'source_id...",ca_fed_house_vote_session_42-43.csv,voting_sessions
4,2018-6-11,Adopted,"Concurrence at report stage and second reading of Bill C-59, An Act respecting national security matters",C-59,"National Security Act, 2017",42,1,https://www.ourcommons.ca/members/en/votes/42/1/747,"[{'name': 'Ziad Aboultaif', 'source_id': '89156', 'vote': 'Nay', 'vote_session_url': 'https://www.ourcommons.ca/members/en/votes/42/1/747', 'bill_number': 'C-59'}, {'name': 'Dan Albas', 'source_id...",ca_fed_house_vote_session_42-43.csv,voting_sessions
